# Provenance of validated defaults: recording overrides

Several omicverse methods have defaults that were validated on independent benchmarks (simulated
generators with known truth, labelled public data). Changing such a default because it looked better
on the data at hand (a simulator built from the same data, a development sweep) usually generalises
worse. `ov.utils.validated_defaults` marks these parameters: an override without `override_reason=`
still runs, but emits a `DefaultOverrideWarning`, is logged for the session and is attached to the
result (`DataFrame.attrs['ov_provenance']`, `AnnData.uns['ov_provenance']`, or a dict key), so a report
or a deliverable checker can see it. The idea follows decision records in computational provenance
(W3C PROV-DM).

In [ ]:
import warnings
import numpy as np, pandas as pd
import omicverse as ov
ov.utils.clear_provenance()
rng = np.random.default_rng(0)
n = 3000
X = pd.DataFrame((rng.random((n, 200)) < 0.3).astype(float), columns=[f'g{i}' for i in range(200)])
y = (rng.random(n) < 1 / (1 + np.exp(-(-2 + 2.5 * X.g0 + 2.0 * X.g1)))).astype(int)

## 1. Defaults kept: nothing is recorded

In [ ]:
res = ov.genetics.trait_determinants(X, y, modifier=None)
print(sorted(res.feature[res.is_determinant]))
print(ov.utils.provenance_report())

['g0', 'g1']
ov provenance: all validated defaults kept.


## 2. Overrides: without a reason they warn and are attached to the result; with `override_reason=` they are only logged

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    res2 = ov.genetics.trait_determinants(X, y, modifier=None, n_pcs=0, select='bic')
print([str(x.message)[:120] for x in w if issubclass(x.category, ov.utils.DefaultOverrideWarning)])
print(res2.attrs['ov_provenance'])
res3 = ov.genetics.trait_determinants(X, y, modifier=None, n_pcs=0,
                                      override_reason='no population structure in this simulated cohort')
print(ov.utils.provenance_report())

["trait_determinants: validated default(s) overridden without override_reason: select='bic' (default 'ebic'), n_pcs=0 (def"]
[{'function': 'omicverse.genetics._determinants.trait_determinants', 'parameter': 'select', 'default': 'ebic', 'value': 'bic', 'reason': None, 'time': '2026-11-23T14:20:04'}, {'function': 'omicverse.genetics._determinants.trait_determinants', 'parameter': 'n_pcs', 'default': 10, 'value': 0, 'reason': None, 'time': '2026-11-23T14:20:04'}]
ov provenance: 2 unexplained / 3 overrides
  omicverse.genetics._determinants.trait_determinants: select='bic' (default 'ebic') -- NO REASON GIVEN
  omicverse.genetics._determinants.trait_determinants: n_pcs=0 (default 10) -- NO REASON GIVEN
  omicverse.genetics._determinants.trait_determinants: n_pcs=0 (default 10) -- no population structure in this simulated cohort


## 3. In a deliverable check

`ov.utils.write_provenance(path)` writes the session log as JSON next to a deliverable; every line marked
**NO REASON GIVEN** should be reverted to the default or justified with independent evidence.

In [ ]:
import tempfile, os, json
p = ov.utils.write_provenance(os.path.join(tempfile.mkdtemp(), 'provenance.json'))
print(json.load(open(p))['overrides'][0]['parameter'], '...', len(json.load(open(p))['overrides']), 'records')

select ... 3 records
